<div style="background: linear-gradient(135deg, #0d0d0d 0%, #1a0025 50%, #0d0011 100%); padding: 30px; border-radius: 15px; text-align: center; border: 2px solid #8b0000; box-shadow: 0 0 30px rgba(139,0,0,0.5);">
  <h1 style="color: #ff4444; font-size: 2.5em; margin: 0; text-shadow: 0 0 20px #ff0000;">&#127908; Elbek Studiya RVC</h1>
  <h2 style="color: #cc44ff; font-size: 1.5em; margin: 5px 0;">Voice Model Trainer</h2>
  <p style="color: #888;">Gradio UI + Ngrok | Bebra777228/TrainVocModel-EN</p>
</div>

In [ ]:
#@title # \U0001f534 **1-BOLIM: O'rnatish** (GPU yoqilgan bo'lishi kerak!)#@markdown ---#@markdown ### Runtime > Change runtime type > **T4 GPU** tanlang!#@markdown ### Keyin shu cellni ishga tushiring. ~6 daqiqa kutasiz.#@markdown ---import os, sys, torchfrom google.colab import drivefrom IPython.display import clear_output, display, HTML# GPUprint("\033[93mGPU tekshirilmoqda...\033[0m")if not torch.cuda.is_available():    raise Exception("\033[91m\n!!! GPU TOPILMADI !!!\n\nRuntime > Change runtime type > T4 GPU tanlang!\033[0m")gpu_name = torch.cuda.get_device_name(0)print(f"\033[92mGPU tayyor: {gpu_name}\033[0m\n")# Driveif not os.path.isdir("/content/drive"):    drive.mount("/content/drive")ROOT_DIR = "/content/drive/MyDrive/ElbekStudiya_RVC"BRANCH = "main"RAW = f"https://raw.githubusercontent.com/Bebra777228/TrainVocModel-EN/refs/heads/{BRANCH}"os.makedirs("/content/dataset", exist_ok=True)# Cloneif not os.path.isdir(ROOT_DIR):    print("Repo clone qilinmoqda...")    os.system(f"git clone https://github.com/Bebra777228/TrainVocModel-EN -b {BRANCH} {ROOT_DIR} &> /dev/null")os.chdir(ROOT_DIR)clear_output()print("\033[93mO'rnatish boshlandi (~6 daqiqa)...\033[0m\n")# Python 3.10print("[1/6] Python 3.10...")os.system("sudo apt update &> /dev/null && sudo apt install python3.10 -y &> /dev/null")os.system("sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1 &> /dev/null")os.system("sudo update-alternatives --set python3 /usr/bin/python3.10 &> /dev/null")os.system("curl -sS https://bootstrap.pypa.io/get-pip.py | python3 &> /dev/null")sys.path.append("/usr/local/lib/python3.10/dist-packages")# Scriptsprint("[2/6] Fayllar yuklanmoqda...")for s in [    "requirements.txt", "download_files.py", "configs/config.py",    "infer/lib/rmvpe.py", "infer/lib/slicer.py", "infer/lib/train/utils.py",    "infer/modules/train/train.py", "infer/modules/vc/infer_cli.py",    "infer/lib/train/process_ckpt.py", "infer/modules/train/extract_f0.py",    "infer/modules/train/preprocess.py", "infer/modules/train/extract_index.py",    "infer/modules/train/extract_feature.py",]:    dest = f"{ROOT_DIR}/{s}"    os.makedirs(os.path.dirname(dest), exist_ok=True)    os.system(f"wget {RAW}/{s} -O {dest} &> /dev/null")# Pipprint("[3/6] Kutubxonalar...")os.system("pip install --upgrade pip==24.0 &> /dev/null")os.system("pip install --no-cache-dir -qq -r requirements.txt &> /dev/null")print("[4/6] aria2, ffmpeg...")os.system("sudo apt install -y aria2 ffmpeg &> /dev/null")print("[5/6] HuBERT modeli...")os.system('python download_files.py "./assets/" "contentvec_base" &> /dev/null')print("[6/6] Gradio + pyngrok...")os.system("pip install -qq gradio==4.44.1 pyngrok &> /dev/null")os.system("rm -rf /content/sample_data")os.system(f"rm -rf {ROOT_DIR}/.git {ROOT_DIR}/notebooks")clear_output()display(HTML(    '<div style="background:#0a1f0a; border:2px solid #00ff00; border-radius:10px; padding:20px; text-align:center;">'    '<h2 style="color:#00ff00;">&#9989; O\'RNATISH TUGADI!</h2>'    '<p style="color:#88ff88; font-size:1.2em;">Endi 2-BOLIMNI ishga tushiring!</p>'    '<p style="color:#88ff88;">ngrok token kiriting va Gradio linkni oling</p>'    '</div>'))

In [ ]:
#@title # \U0001f7e2 **2-BOLIM: Gradio UI + Link olish**#@markdown ---#@markdown ### \U0001f511 Ngrok token kiriting (https://dashboard.ngrok.com/get-started/your-authtoken)#@markdown > Token kiritsangiz **ngrok** link chiqadi.#@markdown >#@markdown > Kirmasangiz **Gradio share** link chiqadi (ham ishlaydi).#@markdown ---ngrok_token = "" #@param {type:"string"}import osimport sysimport globimport jsonimport shutilimport zipfileimport pathlibimport tracebackfrom random import shufflefrom subprocess import PIPE, STDOUT, Popenfrom urllib.parse import urlparsefrom IPython.display import display, HTMLROOT_DIR = "/content/drive/MyDrive/ElbekStudiya_RVC"os.chdir(ROOT_DIR)import gradio as gr# ============================# TRENING FUNKSIYASI# ============================def full_train(model_name, dataset_audio_files, sample_rate, f0_method,               index_algorithm, pretrain_type, epochs, save_epoch,               batch_size, progress=gr.Progress(track_tqdm=True)):    if not model_name or not model_name.strip():        return "\u274c Model nomini kiriting!", "", None    if not dataset_audio_files or len(dataset_audio_files) == 0:        return "\u274c Ovoz fayllarini yuklang!", "", None    model_name = model_name.strip().replace(" ", "_")    logs = []    def log(msg):        logs.append(msg)        print(msg)    try:        os.chdir(ROOT_DIR)        now_dir = os.getcwd()        # 1. DATASET        log("\u2501" * 40)        log("\u25b6 1/6: Dataset tayyorlanmoqda...")        progress(0.05, desc="Dataset tayyorlanmoqda...")        ds_folder = f"/content/dataset_{model_name}"        if os.path.exists(ds_folder):            shutil.rmtree(ds_folder)        os.makedirs(ds_folder)        for af in dataset_audio_files:            shutil.copy2(af, os.path.join(ds_folder, os.path.basename(af)))            log(f"  + {os.path.basename(af)}")        log(f"  Jami: {len(dataset_audio_files)} fayl")        # 2. PREPROCESS        log("\u2501" * 40)        log("\u25b6 2/6: Preprocess...")        progress(0.12, desc="Preprocess (audio qayta ishlash)...")        sr_int = int(sample_rate.rstrip("k")) * 1000        os.makedirs(f"./logs/{model_name}", exist_ok=True)        cmd = f"python {ROOT_DIR}/infer/modules/train/preprocess.py {ds_folder} {sr_int} 2 ./logs/{model_name} False 3.0"        ret = os.system(cmd)        log(f"  Preprocess: {'OK' if ret == 0 else 'XATO (code ' + str(ret) + ')'}")        # 3. F0        log("\u2501" * 40)        log(f"\u25b6 3/6: F0 extraction ({f0_method})...")        progress(0.28, desc=f"F0 ({f0_method})...")        cmd = f"python {ROOT_DIR}/infer/modules/train/extract_f0.py 1 0 0 ./logs/{model_name} False {f0_method}"        ret = os.system(cmd)        log(f"  F0: {'OK' if ret == 0 else 'XATO'}")        # 4. FEATURE        log("\u2501" * 40)        log("\u25b6 4/6: Feature extraction...")        progress(0.42, desc="Feature extraction...")        cmd = f"python {ROOT_DIR}/infer/modules/train/extract_feature.py 1 0 ./logs/{model_name} v2 False"        ret = os.system(cmd)        log(f"  Feature: {'OK' if ret == 0 else 'XATO'}")        # 5. INDEX        log("\u2501" * 40)        log("\u25b6 5/6: Index...")        progress(0.52, desc="Index yaratilmoqda...")        cmd = f"python {ROOT_DIR}/infer/modules/train/extract_index.py ./logs/{model_name} v2 {index_algorithm}"        ret = os.system(cmd)        log(f"  Index: {'OK' if ret == 0 else 'XATO'}")        # 6. PRETRAIN + TRAIN        log("\u2501" * 40)        log(f"\u25b6 6/6: Trening ({epochs} epoch, {pretrain_type})...")        progress(0.58, desc="Pretrain yuklanmoqda...")        param_aria = "--console-log-level=error -c -x 16 -s 16 -k 1M"        hugg = ("ht" + "tps:/" + "/hug" + "gin" + "gfa" + "ce.co" + "/Poli" + "tree"                + "s/RV" + "C_res" + "ourc" + "es/re" + "solv" + "e/ma" + "in/pret"                + "rain" + "ed/v2")        pret_dir = "/content/pretrained_models"        os.system(f"rm -rf {pret_dir}")        os.makedirs(pret_dir)        m = {            "Default": [                (f"{sample_rate}/Default/f0D{sample_rate}.pth", "default_D.pth"),                (f"{sample_rate}/Default/f0G{sample_rate}.pth", "default_G.pth"),            ],            "Snowie v3.1": [                (f"{sample_rate}/Snowie/D_SnowieV3.1_{sample_rate}.pth", "Snowie_D.pth"),                (f"{sample_rate}/Snowie/G_SnowieV3.1_{sample_rate}.pth", "Snowie_G.pth"),            ],            "TITAN-Medium": [                (f"{sample_rate}/TITAN/D-f0{sample_rate}-TITAN-Medium.pth", "TITAN_Medium_D.pth"),                (f"{sample_rate}/TITAN/G-f0{sample_rate}-TITAN-Medium.pth", "TITAN_Medium_G.pth"),            ],            "KLM v4.3 x3": [                (f"{sample_rate}/KLM/D_KLM43_X3_{sample_rate}.pth", "KLM_D.pth"),                (f"{sample_rate}/KLM/G_KLM43_X3_{sample_rate}.pth", "KLM_G.pth"),            ],        }        for f in m[pretrain_type]:            os.system(f"aria2c {param_aria} {hugg}/{f[0]} -d {pret_dir} -o {f[1]} &> /dev/null")        G_file = f"{pret_dir}/{m[pretrain_type][1][1]}"        D_file = f"{pret_dir}/{m[pretrain_type][0][1]}"        log(f"  Pretrain: G={m[pretrain_type][1][1]}, D={m[pretrain_type][0][1]}")        # filelist        f0d = f"{now_dir}/logs/{model_name}/2a_f0"        f0n = f"{now_dir}/logs/{model_name}/2b-f0nsf"        gtw = f"{now_dir}/logs/{model_name}/0_gt_wavs"        ftd = f"{now_dir}/logs/{model_name}/3_feature768"        names = (            set([n.split(".")[0] for n in os.listdir(gtw)])            & set([n.split(".")[0] for n in os.listdir(ftd)])            & set([n.split(".")[0] for n in os.listdir(f0d)])            & set([n.split(".")[0] for n in os.listdir(f0n)])        )        opt = []        for name in names:            opt.append(f"{gtw}/{name}.wav|{ftd}/{name}.npy|{f0d}/{name}.wav.npy|{f0n}/{name}.wav.npy|0")        for _ in range(2):            opt.append(f"{now_dir}/logs/mute/0_gt_wavs/mute{sample_rate}.wav|{now_dir}/logs/mute/3_feature768/mute.npy|{now_dir}/logs/mute/2a_f0/mute.wav.npy|{now_dir}/logs/mute/2b-f0nsf/mute.wav.npy|0")        shuffle(opt)        with open(f"{now_dir}/logs/{model_name}/filelist.txt", "w") as fl:            fl.write("\n".join(opt))        # config        cp = f"configs/v1/{sample_rate}.json" if sample_rate == "40k" else f"configs/v2/{sample_rate}.json"        cs = f"{now_dir}/logs/{model_name}/config.json"        if not os.path.exists(cs):            with open(cp, "r") as cf:                cd = json.load(cf)                cd["train"]["fp16_run"] = True                cd["train"]["log_interval"] = 10            with open(cs, "w", encoding="utf-8") as cf:                json.dump(cd, cf, ensure_ascii=False, indent=4, sort_keys=True)                cf.write("\n")        # TRAIN        progress(0.62, desc=f"Trening: 0/{epochs} epoch...")        train_cmd = (            f"python {ROOT_DIR}/infer/modules/train/train.py "            f"-e '{model_name}' -sr {sample_rate} -bs {batch_size} "            f"-te {epochs} -se {save_epoch} "            f"-pg {G_file} -pd {D_file} -o AdamW"        )        p = Popen(train_cmd, shell=True, cwd=now_dir, stdout=PIPE, stderr=STDOUT,                  bufsize=1, universal_newlines=True)        for line in p.stdout:            line = line.strip()            if line:                log(line)                if "Epoch" in line or "epoch" in line or "E " in line:                    try:                        for pt in line.replace("/", " ").replace(":", " ").split():                            if pt.isdigit():                                ep = int(pt)                                if 0 < ep <= epochs:                                    progress(0.62 + 0.33 * (ep / epochs), desc=f"Trening: {ep}/{epochs}")                                    break                    except:                        pass        p.wait()        log(f"  Train tugadi (code: {p.returncode})")        # ZIP        log("\u2501" * 40)        log("\u25b6 ZIP yaratilmoqda...")        progress(0.96, desc="ZIP yaratilmoqda...")        pth_files = []        wd = f"{ROOT_DIR}/assets/weights"        if os.path.isdir(wd):            pth_files += glob.glob(f"{wd}/{model_name}.pth")            pth_files += glob.glob(f"{wd}/{model_name}_e*.pth")        ld = f"{ROOT_DIR}/logs/{model_name}"        if os.path.isdir(ld):            pth_files += glob.glob(f"{ld}/**/*.pth", recursive=True)        pth_files = list(dict.fromkeys(pth_files))        idx_files = []        if os.path.isdir(ld):            idx_files += glob.glob(f"{ld}/added_{model_name}_v2.index")            idx_files += glob.glob(f"{ld}/added_*.index")            idx_files += glob.glob(f"{ld}/*.index")        idx_files = list(dict.fromkeys(idx_files))        mp = None        if pth_files:            ex = [f for f in pth_files if os.path.basename(f) == f"{model_name}.pth"]            mp = ex[0] if ex else sorted(pth_files)[-1]        mi = idx_files[0] if idx_files else None        zp = f"/content/{model_name}_model.zip"        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:            if mp:                zf.write(mp, os.path.basename(mp))                log(f"  + {os.path.basename(mp)}")            if mi:                zf.write(mi, os.path.basename(mi))                log(f"  + {os.path.basename(mi)}")        zs = os.path.getsize(zp) / (1024 * 1024)        log(f"\n\u2501" * 40)        log(f"\u2705 TAYYOR! ZIP: {zs:.1f} MB")        progress(1.0, desc="\u2705 Tayyor!")        status = f"\u2705 TAYYOR! | .pth: {os.path.basename(mp) if mp else 'yoq'} | .index: {os.path.basename(mi) if mi else 'yoq'} | ZIP: {zs:.1f} MB"        return status, "\n".join(logs), zp    except Exception as e:        err = traceback.format_exc()        logs.append(f"\n\u274c XATO:\n{err}")        return f"\u274c XATO: {e}", "\n".join(logs), None# ============================# GRADIO UI# ============================theme = gr.themes.Base(    primary_hue="red",    secondary_hue="purple",    neutral_hue="gray",)with gr.Blocks(theme=theme, title="Elbek Studiya RVC") as app:    gr.HTML(        '<div style="text-align:center; padding:15px; background:linear-gradient(135deg,#1a0a0a,#2d0a2d); border-radius:12px; border:1px solid #8b0000;">'        '<h1 style="color:#ff4444; margin:0;">&#127908; Elbek Studiya RVC Trainer</h1>'        '<p style="color:#bb88ff; margin:5px 0 0 0;">Ovoz modeli yaratish | Train your voice model</p>'        '</div>'    )    with gr.Tab("\U0001f3a4 Train"):        with gr.Row():            with gr.Column(scale=1):                model_name_in = gr.Textbox(label="\U0001f4dd Model nomi", value="MyVoice", placeholder="MyVoice")                audio_in = gr.File(                    label="\U0001f3b5 Ovoz fayllarini yuklang (WAV, FLAC, MP3)",                    file_count="multiple",                    file_types=["audio"],                    type="filepath",                )                sample_rate_in = gr.Dropdown(["32k", "40k", "48k"], value="40k", label="Sample Rate")                f0_in = gr.Dropdown(["rmvpe", "harvest"], value="rmvpe", label="F0 Method")            with gr.Column(scale=1):                epochs_in = gr.Slider(50, 2000, 500, step=50, label="\U0001f504 Epoch soni")                save_ep_in = gr.Slider(10, 500, 50, step=10, label="Har necha epochda saqlash")                batch_in = gr.Slider(2, 32, 8, step=2, label="Batch Size")                pretrain_in = gr.Dropdown(                    ["Default", "Snowie v3.1", "TITAN-Medium", "KLM v4.3 x3"],                    value="Default", label="Pretrained Model"                )                idx_in = gr.Dropdown(["Auto", "Faiss", "KMeans"], value="Auto", label="Index Algorithm")        train_btn = gr.Button("\U0001f525 TRENING BOSHLASH", variant="primary", size="lg")        status_out = gr.Textbox(label="Holat", interactive=False)        log_out = gr.Textbox(label="Log", interactive=False, lines=12, max_lines=30)        zip_out = gr.File(label="\U0001f4e6 ZIP yuklab olish (.pth + .index)")        train_btn.click(            fn=full_train,            inputs=[model_name_in, audio_in, sample_rate_in, f0_in,                    idx_in, pretrain_in, epochs_in, save_ep_in, batch_in],            outputs=[status_out, log_out, zip_out],        )    with gr.Tab("\u2139\ufe0f Yordam"):        gr.Markdown(            "### Qanday ishlatish:\n"            "1. Model nomini kiriting\n"            "2. Ovoz fayllarini yuklang (WAV/FLAC/MP3, 10+ daqiqa)\n"            "3. Epoch tanlang (500-1000 tavsiya)\n"            "4. TRENING BOSHLASH bosing\n"            "5. Tugaganda ZIP yuklab oling\n\n"            "### Chiqish fayllari:\n"            "- `.pth` - ovoz modeli fayli\n"            "- `.index` - indeks fayli\n\n"            "*Elbek Studiya RVC v7 | Bebra777228/TrainVocModel-EN*"        )# ============================# ISHGA TUSHIRISH# ============================print("\n" + "=" * 55)print("   GRADIO UI ISHGA TUSHMOQDA...")print("=" * 55)ngrok_ok = Falseif ngrok_token and ngrok_token.strip():    try:        from pyngrok import ngrok as ng        ng.set_auth_token(ngrok_token.strip())        public = ng.connect(7860, "http")        pub_url = str(public).replace("NgrokTunnel: ", "").strip().strip('"')        ngrok_ok = True        print(f"\n\033[92m{'=' * 55}")        print(f"   NGROK LINK TAYYOR:")        print(f"   {pub_url}")        print(f"{'=' * 55}\033[0m\n")        print("Shu linkni brauzerda oching!")        display(HTML(            f'<div style="background:#001a00; border:3px solid #00ff00; border-radius:12px; padding:20px; text-align:center; margin:10px 0;">'            f'<h2 style="color:#00ff00;">&#9989; LINK TAYYOR!</h2>'            f'<a href="{pub_url}" target="_blank" style="color:#00ffff; font-size:1.5em; font-weight:bold;">{pub_url}</a>'            f'<p style="color:#88ff88; margin-top:10px;">Shu linkni oching va model qiling!</p>'            f'</div>'        ))        app.launch(server_port=7860, quiet=True, inline=False)    except Exception as e:        print(f"\033[91mNgrok xatosi: {e}\033[0m")        print("Gradio share link ishlatilmoqda...\n")        ngrok_ok = Falseif not ngrok_ok:    print("\nGradio share link yaratilmoqda...\n")    app.launch(share=True, server_port=7860, quiet=False)    print("\n\033[92mYuqoridagi public URL linkni oching!\033[0m")